In [1]:
import torch
import torchvision
import math
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F

device=torch.device("cuda:0")

cfg=dict(
    T=1000, beta_start=1e-4,beta_end=0.02,
    img_size=32, channels=3,
    base_channels=128, channel_mults=(1,2,2,2), num_res_blocks=2, attn_resolution=16, dropout=0.1,
    batch_size=128, lr=2e-4, ema_decay=0.9999,
    sample_every=5000, ckpt_every=5000,
)
torch.manual_seed(0)
print(cfg)
print(torch.cuda.get_device_name())
print(torch.__version__)


{'T': 1000, 'beta_start': 0.0001, 'beta_end': 0.02, 'img_size': 32, 'channels': 3, 'base_channels': 128, 'channel_mults': (1, 2, 2, 2), 'num_res_blocks': 2, 'attn_resolution': 16, 'dropout': 0.1, 'batch_size': 128, 'lr': 0.0002, 'ema_decay': 0.9999, 'sample_every': 5000, 'ckpt_every': 5000}
Tesla T4
2.11.0+cu128


In [ ]:
from torch.utils.data import DataLoader


transforms=torchvision.transforms.Compose(
    [torchvision.transforms.RandomHorizontalFlip(), torchvision.transforms.ToTensor(), torchvision.transforms.Normalize(0.5,0.5)])

dataset=torchvision.datasets.CIFAR10(root="./data",train=True, download=True, transform=transforms)

train_dataloader=DataLoader(dataset, cfg["batch_size"], shuffle=True,num_workers=8, pin_memory=True, drop_last=True)

x0_demo, _=next(iter(train_dataloader))



/home/c504-5090-x2/miniconda3/lib/python3.13/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [2]:
T = cfg["T"]
beta = torch.linspace(cfg["beta_start"], cfg["beta_end"], T, dtype=torch.float64)
alpha = 1.0 - beta
alpha_bar = torch.cumprod(alpha, dim=0)

sqrt_ab   = alpha_bar.sqrt().float().to(device)
sqrt_1mab = (1 - alpha_bar).sqrt().float().to(device)

print(f"ᾱ_1={alpha_bar[0]:.6f}  ᾱ_500={alpha_bar[499]:.4f}  ᾱ_1000={alpha_bar[-1]:.2e}")


ᾱ_1=0.999900  ᾱ_500=0.0786  ᾱ_1000=4.04e-05


In [3]:
def q_sample(x0, t, eps):
    """x0: (B,3,32,32) in [-1,1] | t: (B,) long on device | eps: randn like x0"""
    return (sqrt_ab[t].view(-1, 1, 1, 1) * x0 +
            sqrt_1mab[t].view(-1, 1, 1, 1) * eps)

In [4]:
def timestep_embed(t, dim):
    t=t.float()
    freqs=torch.exp(-math.log(10000)*torch.arange(dim//2, device=t.device)/(dim//2))
    outer_product=t[:,None]*freqs[None,:]
    output=torch.cat([torch.sin(outer_product), torch.cos(outer_product)], dim=-1)
    return output

class TimeEmbedding(nn.Module):
    def __init__(self, base_channels, time_dim):
        super().__init__()
        self.base_channels=base_channels
        self.time_dim=time_dim

        self.linear1=nn.Linear(base_channels, time_dim)
        self.silu=nn.SiLU(inplace=False)
        self.linear2=nn.Linear(time_dim,time_dim)
    def forward(self, t):
        t=timestep_embed(t, self.base_channels)
        t=self.linear1(t)
        t=self.silu(t)
        t=self.linear2(t)
        return t

In [5]:
class ResBlock(nn.Module):
    def __init__(self, time_dim, in_ch, out_ch,cfg):
        super().__init__()
        assert in_ch % 32 == 0 and out_ch % 32 == 0
        self.time_dim=time_dim
        self.out_ch=out_ch
        self.in_ch=in_ch

        self.norm1=nn.GroupNorm(32, in_ch)
        self.silu=nn.SiLU(inplace=False)
        self.conv1=nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1)
        self.linear_proj=nn.Linear(time_dim, 2*out_ch)
        self.norm2=nn.GroupNorm(32, out_ch)
        self.dropout=nn.Dropout(cfg["dropout"])
        self.conv2=nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1)
        if in_ch==out_ch:
            self.res=nn.Identity()
        else:
            self.res=nn.Conv2d(in_ch, out_ch, kernel_size=1)

    def forward(self, x, t_emb):
        h=self.norm1(x)
        h=self.silu(h)
        h=self.conv1(h)

        proj=self.silu(t_emb)
        proj=self.linear_proj(proj)
        scale, shift=proj.chunk(2, dim=1)

        scale=scale.view(-1, self.out_ch, 1 ,1)
        shift=shift.view(-1,self.out_ch, 1, 1)

        h=self.norm2(h)*(1+scale)+shift
        h=self.silu(h)
        h=self.dropout(h)
        h=self.conv2(h)

        return h+self.res(x)







In [6]:
class AttentionBlock(nn.Module):
    def __init__(self, channels, num_heads=None):
        super().__init__()
        if num_heads is None:
            assert channels % 64 == 0
            num_heads = channels // 64          # derive FIRST
        assert channels % num_heads == 0
        self.num_heads = num_heads              # store AFTER deriving
        self.head_dim = channels // num_heads
        self.norm = nn.GroupNorm(32, channels)
        self.proj = nn.Conv2d(channels, channels * 3, 1)
        self.out = nn.Conv2d(channels, channels, 1)

    def forward(self, x):
        h = self.norm(x)
        qkv = self.proj(h)
        q, k, v = qkv.chunk(3, dim=1)
        B, C, H, W = q.shape
        # split channels into heads (legal reshape: splits adjacent axis)...
        q = q.reshape(B, self.num_heads, self.head_dim, H * W).transpose(2, 3)
        k = k.reshape(B, self.num_heads, self.head_dim, H * W).transpose(2, 3)
        v = v.reshape(B, self.num_heads, self.head_dim, H * W).transpose(2, 3)
        # ...then transpose to (B, heads, seq, head_dim) for SDPA
        h = F.scaled_dot_product_attention(q, k, v)
        h = h.transpose(2, 3).reshape(B, C, H, W)   # exact inverse choreography
        h = self.out(h)
        return x + h


In [7]:
class Downsample(nn.Module):
  def __init__(self, channels):
    super().__init__()
    self.conv = nn.Conv2d(channels, channels, 3, stride=2, padding=1)
  def forward(self, x):
    return self.conv(x)

class Upsample(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, 3, padding=1)
    def forward(self, x):
        return self.conv(F.interpolate(x, scale_factor=2, mode="nearest"))


In [8]:
class DownPath(nn.Module):
  def __init__(self, in_channels, base_channels,  time_dim, dropout):
    super().__init__()
    self.skip_chs=[base_channels]*4+[base_channels*2]*8
    self.stem_conv=nn.Conv2d(in_channels, base_channels, kernel_size=3, stride=1, padding=1)
    self.resblock1=ResBlock(time_dim, base_channels, base_channels, cfg)
    self.resblock2=ResBlock(time_dim, base_channels, base_channels, cfg)
    self.downsample1=Downsample(base_channels)

    self.resblock3=ResBlock(time_dim, base_channels, base_channels*2, cfg)
    self.attn1=AttentionBlock(base_channels*2)

    self.resblock4=ResBlock(time_dim, base_channels*2, base_channels*2, cfg)
    self.attn2=AttentionBlock(base_channels*2)

    self.downsample2=Downsample(base_channels*2)

    self.resblock5=ResBlock(time_dim, base_channels*2, base_channels*2, cfg)
    self.resblock6=ResBlock(time_dim, base_channels*2, base_channels*2, cfg)

    self.downsample3=Downsample(base_channels*2)

    self.resblock7=ResBlock(time_dim, base_channels*2, base_channels*2, cfg)
    self.resblock8=ResBlock(time_dim, base_channels*2, base_channels*2, cfg)
  def forward(self, x, t_emb):
    hs=[]
    h=self.stem_conv(x)
    hs.append(h)

    h=self.resblock1(h, t_emb)
    hs.append(h)
    h=self.resblock2(h, t_emb)
    hs.append(h)

    h=self.downsample1(h)
    hs.append(h)

    h=self.resblock3(h, t_emb)
    h=self.attn1(h)
    hs.append(h)

    h=self.resblock4(h,t_emb)
    h=self.attn2(h)
    hs.append(h)

    h=self.downsample2(h)
    hs.append(h)

    h=self.resblock5(h, t_emb)
    hs.append(h)
    h=self.resblock6(h, t_emb)
    hs.append(h)

    h=self.downsample3(h)
    hs.append(h)

    h=self.resblock7(h, t_emb)
    hs.append(h)
    h=self.resblock8(h, t_emb)
    hs.append(h)

    return h, hs




In [9]:
down = DownPath(3, 128, 512, 0.1).to(device)   # adjust to your final signature
x  = torch.randn(2, 3, 32, 32, device=device)
te = torch.randn(2, 512, device=device)
h, hs = down(x, te)
print(h.shape)                                  # (2, 256, 4, 4)
print(len(hs))                                  # 12
print([tuple(t.shape[1:]) for t in hs])         # channel/res of every push
print(down.skip_chs)                            # [128,128,128,128,256,...,256]

torch.Size([2, 256, 4, 4])
12
[(128, 32, 32), (128, 32, 32), (128, 32, 32), (128, 16, 16), (256, 16, 16), (256, 16, 16), (256, 8, 8), (256, 8, 8), (256, 8, 8), (256, 4, 4), (256, 4, 4), (256, 4, 4)]
[128, 128, 128, 128, 256, 256, 256, 256, 256, 256, 256, 256]


In [12]:
class UpPath(nn.Module):
  def __init__(self, base_channels, time_dim, dropout):
    super().__init__()
    self.resblock1=ResBlock(time_dim, base_channels*4, base_channels*2, cfg)
    self.resblock2=ResBlock(time_dim, base_channels*4, base_channels*2, cfg)
    self.resblock3=ResBlock(time_dim, base_channels*4, base_channels*2, cfg)
    self.upsample1=Upsample(base_channels*2)

    self.resblock4=ResBlock(time_dim, base_channels*4, base_channels*2, cfg)
    self.resblock5=ResBlock(time_dim, base_channels*4, base_channels*2, cfg)
    self.resblock6=ResBlock(time_dim, base_channels*4, base_channels*2, cfg)
    self.upsample2=Upsample(base_channels*2)

    self.resblock7=ResBlock(time_dim, base_channels*4, base_channels*2, cfg)
    self.attn1=AttentionBlock(base_channels*2)

    self.resblock8=ResBlock(time_dim, base_channels*4, base_channels*2, cfg)
    self.attn2=AttentionBlock(base_channels*2)

    self.resblock9=ResBlock(time_dim, base_channels*3, base_channels*2, cfg)
    self.attn3=AttentionBlock(base_channels*2)

    self.upsample3=Upsample(base_channels*2)
    self.resblock10=ResBlock(time_dim, base_channels*3, base_channels, cfg)
    self.resblock11=ResBlock(time_dim, base_channels*2, base_channels, cfg)
    self.resblock12=ResBlock(time_dim, base_channels*2, base_channels, cfg)


  def forward(self, h, hs, t_emb):
    h=torch.cat([h, hs.pop()], dim=1)
    h=self.resblock1(h, t_emb)
    h=torch.cat([h, hs.pop()], dim=1)
    h=self.resblock2(h, t_emb)
    h=torch.cat([h, hs.pop()], dim=1)
    h=self.resblock3(h, t_emb)
    h=self.upsample1(h)

    h=torch.cat([h, hs.pop()], dim=1)
    h=self.resblock4(h, t_emb)
    h=torch.cat([h, hs.pop()], dim=1)
    h=self.resblock5(h, t_emb)
    h=torch.cat([h, hs.pop()], dim=1)
    h=self.resblock6(h, t_emb)
    h=self.upsample2(h)

    h=torch.cat([h, hs.pop()], dim=1)
    h=self.resblock7(h, t_emb)
    h=self.attn1(h)
    h=torch.cat([h, hs.pop()], dim=1)
    h=self.resblock8(h, t_emb)
    h=self.attn2(h)
    h=torch.cat([h, hs.pop()], dim=1)
    h=self.resblock9(h, t_emb)
    h=self.attn3(h)
    h=self.upsample3(h)

    h=torch.cat([h, hs.pop()], dim=1)
    h=self.resblock10(h, t_emb)
    h=torch.cat([h, hs.pop()], dim=1)
    h=self.resblock11(h, t_emb)
    h=torch.cat([h, hs.pop()], dim=1)
    h=self.resblock12(h, t_emb)

    assert len(hs)==0
    return h



In [13]:
up = UpPath(128, 512, 0.1).to(device)
h, hs = down(x, te)                    # reuse the verified DownPath outputs
out = up(h, hs, te)
print(out.shape)                       # (2, 128, 32, 32)
print(len(hs))                        # 0 -- consumed in place

torch.Size([2, 128, 32, 32])
0


In [41]:
class Middle(nn.Module):
  def __init__(self, base_channels, time_dim, cfg):
    super().__init__()
    self.mid_block1=ResBlock(time_dim, base_channels*2, base_channels*2, cfg)
    self.mid_attn=AttentionBlock(2*base_channels)
    self.mid_block2=ResBlock(time_dim, base_channels*2, base_channels*2, cfg)
  def forward(self, x, t_emb):
    x=self.mid_block1(x, t_emb)
    x=self.mid_attn(x)
    x=self.mid_block2(x, t_emb)
    return x


In [42]:
class Head(nn.Module):
  def __init__(self, base_channels):
    super().__init__()
    self.norm1=nn.GroupNorm(32, base_channels)
    self.silu=nn.SiLU(inplace=False)
    self.conv1=nn.Conv2d(base_channels, 3, kernel_size=3, padding=1)
    nn.init.zeros_(self.conv1.weight)
    nn.init.zeros_(self.conv1.bias)
  def forward(self, x):
    x=self.norm1(x)
    x=self.silu(x)
    x=self.conv1(x)
    return x

In [43]:
class UNet(nn.Module):
  def __init__(self, in_channels, base_channels, time_dim, dropout):
    super().__init__()
    self.time_embedding=TimeEmbedding(base_channels, time_dim)
    self.down=DownPath(in_channels, base_channels, time_dim, dropout)
    self.middle=Middle(base_channels, time_dim, cfg)
    self.up=UpPath(base_channels, time_dim, dropout)
    self.head=Head(base_channels)

  def forward(self, x, t):
    t_emb=self.time_embedding( t)
    h, hs=self.down(x, t_emb)
    h=self.middle(h, t_emb)
    h=self.up(h, hs, t_emb)
    h=self.head(h)
    return h



In [44]:
model = UNet(3, 128, 512, 0.1).to(device)

# (a) param count
n = sum(p.numel() for p in model.parameters())
print(f"{n/1e6:.2f}M")                          # expecting ~35-36M

# (b) forward shape
xb = torch.randn(8, 3, 32, 32, device=device)
tb = torch.randint(0, 1000, (8,), device=device)
out = model(xb, tb)
print(out.shape)                                # (8, 3, 32, 32)

# (c) zero-init head proof
print(out.abs().max().item())                   # exactly 0.0

# (d) full gradient sweep
model(xb, tb).sum().backward()   # fresh forward, since (c) was pre-backward... fine either way
missing = [n_ for n_, p in model.named_parameters() if p.grad is None]
print("no-grad params:", missing)               # must be []

38.31M
torch.Size([8, 3, 32, 32])
0.0
no-grad params: []


In [47]:
class EMA:
  def __init__(self, model, decay):
    self.decay=decay
    self.shadow={n: p.detach().clone() for n, p in model.named_parameters()}

  @torch.no_grad()
  def update(self, model):
    for n, p in model.named_parameters():
      self.shadow[n].mul_(self.decay).add_(p, alpha=1-self.decay)

  @torch.no_grad()
  def copy_to(self, model):
    for n, p in model.named_parameters():
      p.copy_(self.shadow[n])

In [48]:
model2 = UNet(3, 128, 512, 0.1).to(device)
ema = EMA(model2, 0.9999)
opt = torch.optim.Adam(model2.parameters(), lr=1e-3)
for _ in range(3):
    model2(xb, tb).sum().backward(); opt.step(); opt.zero_grad(); ema.update(model2)
name0 = next(iter(ema.shadow))
assert not torch.equal(ema.shadow[name0], dict(model2.named_parameters())[name0])

ema0 = EMA(model2, 0.0)
ema0.update(model2)
assert all(torch.equal(ema0.shadow[n], p) for n, p in model2.named_parameters())
print("EMA verified")

EMA verified


In [49]:
def loss_fn(model, x0):
  t=torch.randint(0, T, (x0.shape[0],), device=x0.device)
  eps=torch.randn_like(x0)
  x_t=q_sample(x0, t, eps)
  return F.mse_loss(model(x_t, t), eps)

In [50]:
model.eval()
with torch.no_grad():
    losses = [loss_fn(model, x0_demo.to(device)).item() for _ in range(10)]
import numpy as np
print(f"init loss: {np.mean(losses):.4f} ± {np.std(losses):.4f}")
# expect: 1.00 ± ~0.01 — the zero-init head predicts 0, so loss = E[ε²] = Var(N(0,1)) = 1
model.train()

NameError: name 'x0_demo' is not defined